In [2]:
!ls

07e5a54c8103b08c542e78c44428e982.genes.results
0c525055a000db7107e1820e5199e5eb_T_rnaseq_Ashion.genes.results
37eabd98258ef3a6ae37726cf6ebef26_T_rnaseq_Ashion.genes.results
3d8ec4f85a02129438c55adbb220cc0b.genes.results
3fa5cb8fabb4944626fffc7ab8092e88_T_rnaseq_Ashion.genes.results
41702a870ac457e219dedd583718e396_T_rnaseq_Ashion.genes.results
5ec594ab1aecfe18e7db699d3b38fece.genes.results
5ec594ab1aecfe18e7db699d3b38fece_T_rnaseq_Ashion_small.genes.results
85c8313f4a40f19524e362aaed6b3f80_T_rnaseq_Ashion.genes.results
89359dd8e3fe3d7dc7aeb8af4e9595ae_T_rnaseq_Ashion.genes.results
94ea945021e65e64388120541f38c522.genes.results
Instructions.ipynb
SAMPLE_INFO.txt
a5e8565594b34d5b9fee74aeb975754f_T_rnaseq_Ashion.genes.results
b254a92b2864c2e3fd61b282a96bd788.genes.results
b3c594f6d14a6cfa9b2649d1a4f0c6a0.genes.results
bd4da66ec03fcdabba9b1fade079a5e7.genes.results
c7c6e94dc0198e02cb6d6f2baab664b4.genes.results
convert_ENSG_to_Symbol.ipynb
d04d55e88308306af3ea10b2ab14a1ef.genes.results
pcp

In [2]:
with open('07e5a54c8103b08c542e78c44428e982.genes.results') as f:
    for line in f:
        if line.startswith('ENSG'):
            ensg = line.split('\t')[0].split('.')[0]
            print(ensg)
            break

ENSG00000000003


In [10]:
!pip install pandas

     |████████████████████████████████| 9.5 MB 16.2 MB/s            
     |████████████████████████████████| 14.8 MB 54.9 MB/s            


In [14]:
# Read the file as a Pandas DataFrame
import pandas as pd
df = pd.read_csv('07e5a54c8103b08c542e78c44428e982.genes.results', sep='\t')
df.head()

,gene_id,transcript_id(s),length,effective_length,expected_count,TPM,FPKM
0,ENSG00000000003.14,"ENST00000373020.8,ENST00000494424.1,ENST000004...",2454.65,2300.93,237.00,9.13,7.80
1,ENSG00000000005.6,"ENST00000373031.5,ENST00000485971.1",823.68,669.97,3.00,0.40,0.34
2,ENSG00000000419.12,"ENST00000371582.8,ENST00000371584.8,ENST000003...",1075.00,921.28,76.00,7.31,6.24
3,ENSG00000000457.14,"ENST00000367770.5,ENST00000367771.11,ENST00000...",4747.51,4593.80,186.06,3.59,3.07
4,ENSG00000000460.17,"ENST00000286031.10,ENST00000359326.9,ENST00000...",3146.51,2992.79,134.94,4.00,3.41


In [15]:
# Clear the gene_id column
df['gene_id_clean'] = df['gene_id'].str.replace(r'\..*$', '', regex=True)
df[['gene_id', 'gene_id_clean']].head()

,gene_id,gene_id_clean
0,ENSG00000000003.14,ENSG00000000003
1,ENSG00000000005.6,ENSG00000000005
2,ENSG00000000419.12,ENSG00000000419
3,ENSG00000000457.14,ENSG00000000457
4,ENSG00000000460.17,ENSG00000000460


In [17]:
!pip install mygene

In [18]:
# Install and use mygene library to convert ENSG to symbols
import mygene

# Create mygene
mg = mygene.MyGeneInfo()

# Get list of clean ENSG
ensg_list = df['gene_id_clean'].tolist()

# Check all the ENSGs on mygene
results = mg.querymany(ensg_list, scopes='ensembl.gene', fields='symbol', species='human')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
69 input query terms found dup hits:	[('ENSG00000002586', 2), ('ENSG00000124333', 2), ('ENSG00000124334', 2), ('ENSG00000167393', 2), ('E
1278 input query terms found no hit:	['ENSG00000112096', 'ENSG00000116883', 'ENSG00000130489', 'ENSG00000130723', 'ENSG00000131484', 'ENS


In [19]:
results[:5]

[{'query': 'ENSG00000000003',
  '_id': '7105',
  '_score': 32.82971,
  'symbol': 'TSPAN6'},
 {'query': 'ENSG00000000005',
  '_id': '64102',
  '_score': 32.82955,
  'symbol': 'TNMD'},
 {'query': 'ENSG00000000419',
  '_id': '8813',
  '_score': 31.813084,
  'symbol': 'DPM1'},
 {'query': 'ENSG00000000457',
  '_id': '57147',
  '_score': 32.829575,
  'symbol': 'SCYL3'},
 {'query': 'ENSG00000000460',
  '_id': '55732',
  '_score': 32.82971,
  'symbol': 'FIRRM'}]

In [20]:
# Convert the results to a DataFrame and join them together
symbol_df = pd.DataFrame(results)

In [21]:
symbol_df[:5]

,query,_id,_score,symbol,notfound
0,ENSG00000000003,7105,32.829710,TSPAN6,NaN
1,ENSG00000000005,64102,32.829550,TNMD,NaN
2,ENSG00000000419,8813,31.813084,DPM1,NaN
3,ENSG00000000457,57147,32.829575,SCYL3,NaN
4,ENSG00000000460,55732,32.829710,FIRRM,NaN


In [22]:
# Merge with the original DataFrame using 'gene_id_clean'
df_with_symbols = df.merge(symbol_df, left_on='gene_id_clean', right_on='query')

In [23]:
df_with_symbols[:5]

,gene_id,transcript_id(s),length,effective_length,expected_count,TPM,FPKM,gene_id_clean,query,_id,_score,symbol,notfound
0,ENSG00000000003.14,"ENST00000373020.8,ENST00000494424.1,ENST000004...",2454.65,2300.93,237.00,9.13,7.80,ENSG00000000003,ENSG00000000003,7105,32.829710,TSPAN6,NaN
1,ENSG00000000005.6,"ENST00000373031.5,ENST00000485971.1",823.68,669.97,3.00,0.40,0.34,ENSG00000000005,ENSG00000000005,64102,32.829550,TNMD,NaN
2,ENSG00000000419.12,"ENST00000371582.8,ENST00000371584.8,ENST000003...",1075.00,921.28,76.00,7.31,6.24,ENSG00000000419,ENSG00000000419,8813,31.813084,DPM1,NaN
3,ENSG00000000457.14,"ENST00000367770.5,ENST00000367771.11,ENST00000...",4747.51,4593.80,186.06,3.59,3.07,ENSG00000000457,ENSG00000000457,57147,32.829575,SCYL3,NaN
4,ENSG00000000460.17,"ENST00000286031.10,ENST00000359326.9,ENST00000...",3146.51,2992.79,134.94,4.00,3.41,ENSG00000000460,ENSG00000000460,55732,32.829710,FIRRM,NaN


In [24]:
total_genes = len(df_with_symbols)
print(f"Total genes: {total_genes}")

Total genes: 58990


In [25]:
mapped = df_with_symbols['symbol'].notna().sum()
print(f"Genes with assigned symbol: {mapped}")

Genes with assigned symbol: 44624


In [26]:
not_found_df = len(df_with_symbols[df_with_symbols['notfound'] == True])
print(f"Genes not found: {not_found_df}")

Genes not found: 1280


In [27]:
# View first results
df_with_symbols[['gene_id', 'gene_id_clean', 'symbol', 'TPM']].head()

,gene_id,gene_id_clean,symbol,TPM
0,ENSG00000000003.14,ENSG00000000003,TSPAN6,9.13
1,ENSG00000000005.6,ENSG00000000005,TNMD,0.40
2,ENSG00000000419.12,ENSG00000000419,DPM1,7.31
3,ENSG00000000457.14,ENSG00000000457,SCYL3,3.59
4,ENSG00000000460.17,ENSG00000000460,FIRRM,4.00


In [28]:
df_mapped_with_symbols = df_with_symbols[df_with_symbols['symbol'].notna()]

In [18]:
df_mapped_with_symbols[['gene_id', 'gene_id_clean', 'symbol', 'TPM']].to_csv('TPM_with_gene_symbols.tsv', sep='\t', index=False)

In [19]:
# DataFrame arranged alphabetically by symbol
df_mapped_with_symbols_sorted = df_mapped_with_symbols.sort_values(by='symbol')

In [20]:
print(df_mapped_with_symbols_sorted.head())

                 gene_id   transcript_id(s)  length  effective_length  \
54573  ENSG00000277488.1  ENST00000618379.1    87.0              0.00   
54530  ENSG00000277411.1  ENST00000614916.1   106.0              0.00   
54270  ENSG00000276861.1  ENST00000612131.1   110.0              0.00   
19421  ENSG00000202198.1  ENST00000365328.1   331.0            177.48   
54490  ENSG00000277313.1  ENST00000622040.1   250.0             97.73   

       expected_count     TPM    FPKM    gene_id_clean            query  \
54573            0.00    0.00    0.00  ENSG00000277488  ENSG00000277488   
54530            0.00    0.00    0.00  ENSG00000277411  ENSG00000277411   
54270            0.00    0.00    0.00  ENSG00000276861  ENSG00000276861   
19421          272.62  136.13  116.28  ENSG00000202198  ENSG00000202198   
54490            0.00    0.00    0.00  ENSG00000277313  ENSG00000277313   

                   _id     _score   symbol notfound  
54573  ENSG00000277488  32.829346  5S_rRNA      NaN  
54

In [21]:
df_mapped_with_symbols_sorted[['gene_id', 'gene_id_clean', 'symbol', 'TPM']].to_csv('Sorted_TPM_with_gene_symbols.tsv', sep='\t', index=False)

In [34]:
# Create a DataFrame without TPM column
df_without_TPM_column = df_mapped_with_symbols[['gene_id', 'gene_id_clean', 'symbol']].sort_values(by='symbol')
print(df_without_TPM_column.head())

                 gene_id    gene_id_clean   symbol
54573  ENSG00000277488.1  ENSG00000277488  5S_rRNA
54530  ENSG00000277411.1  ENSG00000277411  5S_rRNA
54270  ENSG00000276861.1  ENSG00000276861  5S_rRNA
19421  ENSG00000202198.1  ENSG00000202198      7SK
54490  ENSG00000277313.1  ENSG00000277313      7SK


In [8]:
# Merge files that contain columns like gene_id and TPM
import os
import pandas as pd

# Folder where all the files are
files = sorted([f for f in os.listdir('.') if f.startswith('rady') and f.endswith('.results')])
print(files)

['rady_069.genes.results', 'rady_070.genes.results', 'rady_071.genes.results', 'rady_074.genes.results', 'rady_075.genes.results', 'rady_076.genes.results', 'rady_078.genes.results', 'rady_079.genes.results', 'rady_080.genes.results']


In [12]:
label = os.path.splitext(files[0])
label = os.path.splitext(label[0])
print(label[0])

rady_069


In [35]:
# DataFrame
master_df = None

# Iterate over all files
for file in files:
    # Read file
    df = pd.read_csv(file, sep='\t')
    
    # Create column
    df['gene_id_clean'] = df['gene_id'].str.replace(r'\..*$', '', regex=True)
    
    # Extract gene_id_clean and TPM
    df_tpm = df[['gene_id_clean', 'TPM']].copy()
    
    # Rename TPM column
    file_label = os.path.splitext(file) # Remove .results
    file_label = os.path.splitext(file_label[0])[0] # Remove .genes
    df_tpm.rename(columns={'TPM': f'TPM_{file_label}'}, inplace=True)
    
    # Merge to the master DataFrame
    if master_df is None:
        master_df = df_tpm
    else:
        master_df = master_df.merge(df_tpm, on='gene_id_clean', how='outer')

# Final merge with df_without_TPM_column
final_df = df_without_TPM_column.merge(master_df, on='gene_id_clean', how='left')

# Export .csv
final_df.to_csv('TPM_combined_rady_files.tsv', sep='\t', index=False)